# Red neuronal recurrente (RNN) para valorar la call asiática

La red recibe la trayectoria de WMT, MRK y NVDA en los 20 trimestres y predice el payoff de la opción. Aprende de las 100.000 simulaciones de montecarlo.ipynb.

## 1. Cargar los resultados del Montecarlo

montecarlo.ipynb guarda en `Datos/resultados_montecarlo.npz` las trayectorias simuladas (`x`) y el payoff de cada simulación.

In [1]:
import numpy as np

datos = np.load("Datos/resultados_montecarlo.npz")
X, payoff = datos["x"], datos["payoff"]
print("X:", X.shape, "-> (simulaciones, trimestres, acciones)")
print("payoff:", payoff.shape, "-> uno por simulación")

X: (100000, 20, 3) -> (simulaciones, trimestres, acciones)
payoff: (100000,) -> uno por simulación


## 2. Entrenamiento, validación y test

Repartimos las simulaciones al azar: 80 % para entrenar, 10 % para validación y 10 % para test. La entrada es log(X), porque los precios van de 0,01 a 50 y en logaritmo quedan cerca de 0. La salida es el payoff.

In [2]:
import torch
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(0)

X_t = torch.tensor(np.log(X), dtype=torch.float32)               # (100000, 20, 3)
Y_t = torch.tensor(payoff, dtype=torch.float32).unsqueeze(-1)    # (100000, 1)

idx = torch.randperm(len(X_t))
n_train, n_val = int(0.8 * len(idx)), int(0.1 * len(idx))
train, val, test = idx[:n_train], idx[n_train:n_train + n_val], idx[n_train + n_val:]

dl_train = DataLoader(TensorDataset(X_t[train], Y_t[train]), batch_size=256, shuffle=True)
dl_val   = DataLoader(TensorDataset(X_t[val],   Y_t[val]),   batch_size=1024)
dl_test  = DataLoader(TensorDataset(X_t[test],  Y_t[test]),  batch_size=1024)

print("train:", len(train), "| val:", len(val), "| test:", len(test))

train: 80000 | val: 10000 | test: 10000
